### Overview 

This tutorial will go over how to load the BCI data and access its contents 

### Import required packages 

The BCI data is packaged in NWB format with a Zarr backend. To access the data, use `NWBZarrIO` from `hdmf-zarr`. `nwb2widget` creates an interactive GUI with the NWB file, which is useful for exploring the file contents and looking at basic plots.

In [1]:
from hdmf_zarr import NWBZarrIO
from nwbwidgets import nwb2widget
import numpy as np

### Load the data

Let's load the data for one recording session using `NWBZarrIO`. 

In [2]:
# Set filename 
nwb_path = '/data/brain-computer-interface/single-plane-ophys_731015_2025-01-10_18-06-31_processed_2025-08-03_20-39-09/single-plane-ophys_731015_2025-01-10_18-06-31_behavior_nwb' 

# Assign file to an NWBZarrIO object 
io = NWBZarrIO(nwb_path, 'r')
# Read the file 
nwbfile = io.read()

/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'core' version 2.7.0 because version 2.8.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."


`nwb2widget` is useful for exploring the NWB file structure and contents. The widget can also generate basic plots, like the neural activity timeseries traces.  

In [3]:
nwb2widget(nwbfile) 

#### Image Segmentation Table 
    
During processing, the raw fluorescence data is run through a segmentation algorithm (Suite2p, Cellpose) that extracts ROIs for detected neurons, yielding image masks (HxW sparse array with non-zero values where the ROI is in the imaging plane). The outputs are run through an additional soma/dendrite classifier. The image masks and outputs of the soma/dendrite classifier are stored in the `image_segmentation` table in the `processing` container.
    
| Column    | Description |
| -------- | ------- |
| is_soma  | ==1 if ROI classified as soma, ==0 if not  |
| soma_probability | if >0.5 classified as soma  |
| is_dendrite |  ==1 if ROI classified as dendrite, ==0 if not   |
| dendrite_probability   |  if >0.5 classified as dendrite  |
| image_mask  | HxW sparse array defining image masks|

The tables are stored in DynamicTable format. For ease of use, we'll convert the table into a pandas DataFrame. 

In [4]:
image_segmentation = nwbfile.processing["processed"].data_interfaces["image_segmentation"].plane_segmentations["roi_table"].to_dataframe()

#### Cell activity traces 

After the ROIs are extracted, the change in fluorescence over a baseline (dff) is calculated for each ROI. The dff data is accessible as shown below.

The shape of dff is nframes x nrois. 

In [5]:
dff = nwbfile.processing["processed"].data_interfaces["dff"].roi_response_series["dff"].data

print('dff shape (nframes, nrois):',np.shape(dff))

frame_rate = nwbfile.imaging_planes["processed"].imaging_rate
print('Frame Rate:', frame_rate)

dff shape (nframes, nrois): (220344, 1214)
Frame Rate: 58.2634


#### Experiment Structure 
    
The dff array covers the entire experimental period, which has 5 stimulus epochs*. 

    1. Photostimulation of single neurons 
    2. Spontaneous activity 
    3. BCI behavior task 
    4. Spontaneous activity 
    5. Photostimulation of single neurons 
    
*The epoch order is variable across sessions. Sometimes the spontaneous epoch occurs before photostimulation and sometimes there are repeats of the same epoch. Check the epoch table to get the epoch structure for that session. 

#### Epoch Table 
    
The epoch table contains the start and stop times/frames for each stimulus epoch. You can use the epoch table with the dff array to pull and compare neural activity across different stimulus epochs. 

| Column    | Description |
| -------- | ------- |
| stimulus_name  | descriptive name of the epoch  |
| start_frame | epoch start(frames)   |
| stop_frame | epoch end (frames)     |
| start_time    | epoch start (sec)  |
| stop_time   | epoch end (sec)  |

The epoch table is stored as a DynamicTable. For ease of use, convert the DynamicTable into a pandas DataFrame. 

In [6]:
epoch_table = nwbfile.intervals["epochs"].to_dataframe()
epoch_table

,stimulus_name,start_frame,stop_frame,start_time,stop_time
id,,,,,
0,spont,0,2399,0.000000,41.175077
1,photostim,2400,43793,41.192241,751.638250
2,spont_01,43794,49519,751.655413,849.916071
3,BCI,49520,89755,849.933234,1540.503987
4,photostim_post,89756,220343,1540.521150,3781.842460


#### Photostimulation Table  

During the "photostim" epochs, single neurons were optogenetically activated using 2p photostimulation to probe the functional connectivity in the network. 

The PhotostimTrials table (stimulus>PhotostimTrials) contains information about the photostimulation trials. 

| Column    | Description |
| -------- | ------- |
| start_time  | stimulus start (s)  |
| stop_time | stimulus end (s)   |
| start_frame | stimulus start (frame)     |
| stop_frame    | stimulus end (frame)  |
| tiff_file   | data source file name  |
| stimulus_name    | stimulus name   |
| laser_x    | x coordinate of stimulated neuron (pixels)   |
| laser_y    | y coordinate of stimulated neuron (pixels)  |
| power    | stimulus intensity (mW)  |
| duration    | trial duration (s)  |
| stimulus_function    | stimulus template   |
| group_index    | number identifier for stimulated neuron(s)   |
| closest_roi    | index in dff that corresponds to the photostimulated neuron   |


In [7]:
photostim = nwbfile.stimulus["PhotostimTrials"].to_dataframe()

#### BCI Behavior Table 
    
During the "BCI" epochs, the mouse engaged in an optical brain-computer-interface task in which the activity of a single neuron in the imaging plane was used to control the movement of a reward lickport towards its face. 

Information about each BCI behavior trial can be found in the intervals > trials table. 

| Column    | Description |
| -------- | ------- |
| start_time  | trial start (sec)  |
| stop_time | trial end (sec)   |
| go_cue |  time of go cue relative to start time (sec)   |
| hit   |  boolean of whether trial was hit   |
| lick_l  | lick times (sec)   |
| reward_time   | reward delivery time (sec)   |
| threshold_crossing_times    | time when reward port crossed position threshold (sec)   |
| zaber_steps_times   | position of reward port  |
| tiff_file    | data source file  |
| start_frame    | trial start (frame)  |
| stop_frame    | trial end (frame)  |
| conditioned_neuron_x    | coordinate for conditioned neuron (pixels)  |
| conditioned_neuron_y    | coordinate for conditioned neuron (pixels)  |
| closest_roi    | index in dff that corresponds to the photostimulated neuron  |


In [8]:
bci = nwbfile.stimulus["Trials"].to_dataframe()

In [15]:
bci

,start_time,stop_time,go_cue,hit,lick_L,reward_time,threshold_crossing_times,zaber_step_times,tiff_file,start_frame,stop_frame,conditioned_neuron_x,conditioned_neuron_y,closest_roi
id,,,,,,,,,,,,,,
0,849.933234,864.247538,0.2359,True,"[5.2544, 5.4253, 5.5753, 5.682, 5.793, 5.93950...",5.2544,5.1198,"[3.6918, 4.5068, 4.7348, 4.8018, 4.8368, 4.868...",neuron46_00001.tif,49520,50354,56.5,112.5,38
1,864.264701,878.046939,0.2359,True,"[6.526000000000001, 6.6604, 6.7766, 6.9938, 7....",6.5260,6.4260,"[0.5134000000000001, 0.6654, 1.1373, 1.4703000...",neuron46_00002.tif,50355,51158,56.5,112.5,38
2,878.064102,884.672024,0.2359,True,"[1.0756, 1.1934, 1.3018, 1.4094, 1.5212, 1.641...",1.0756,0.8314,"[0.2868, 0.3497, 0.4128, 0.4757, 0.5298, 0.572...",neuron46_00003.tif,51159,51544,56.5,112.5,38
3,884.689187,889.786727,0.2359,True,"[1.7614, 1.9537, 2.0812, 2.3348, 2.4525, 2.563...",1.9537,1.8503,"[0.7273000000000001, 0.8013, 0.862300000000000...",neuron46_00004.tif,51545,51842,56.5,112.5,38
4,889.803891,894.935757,0.2359,True,"[1.8142, 1.9819, 2.142, 2.4023000000000003, 2....",1.9819,1.9437,"[0.2868, 0.3557, 0.4288, 0.4937, 0.7277, 0.817...",neuron46_00005.tif,51843,52142,56.5,112.5,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,1488.121188,1497.029010,0.2359,True,"[5.7352, 5.8565000000000005, 6.0655, 6.1877, 6...",5.7352,5.6937,"[0.317, 2.4249, 4.7477, 5.0557, 5.1467, 5.2097...",neuron46_00061.tif,86703,87222,56.5,112.5,38
61,1497.046173,1509.077740,0.2359,True,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",NaN,NaN,"[1.0342, 6.205900000000001, 6.9289000000000005...",neuron46_00062.tif,87223,87924,56.5,112.5,38
62,1509.094903,1521.126470,0.2359,True,"[9.5475, 9.7215, 9.8932, 10.0676, 10.5599, nan...",NaN,NaN,"[3.8384, 6.727200000000001, 6.8802, 6.9702, 7....",neuron46_00063.tif,87925,88626,56.5,112.5,38
